# Suite de Evaluación RWKV 2.9B Merged (Kaggle 2x T4)

- **Checkpoint**: Merged rearmado en CPU a partir de `BlinkDL/rwkv7-g1` (`rwkv7-g1j-2.9b-20260831-ctx16384.pth`) y el adapter LoRA `out/kateto-final/` (`adapter_model.safetensors` + `adapter_config.json`).
- **Device / Machine Shape**: `NvidiaTeslaT4x2`
- **Motor**: `rwkv-peft` nativo (`.pth`) / `rwkvt.rwkv7.model.RWKV7`, `WKV=triton`.

### Definiciones fijadas (§1)
- `prompt_sha256`: SHA-256 hex del texto del prompt tal cual (UTF-8, sin andamiaje de formato).
- `turno`:
  - `frio`: primera generación de ese prompt en el proceso.
  - `caliente`: la MISMA llamada repetida inmediatamente después, con `torch.manual_seed(1337)` reseteado antes de cada una (mismo texto, cambia solo el tiempo).
- `ttfab_s`: segundos de pared desde el arranque de la generación hasta el PRIMER token muestreado (incluye prefill y, en la primera llamada del proceso, la compilación de kernels triton). Medido envolviendo `rwkv_pipeline.opd_rollout._next_logits`.
- `frio_proceso`: fila extra con la primerísima llamada tras cargar el modelo (prompt dummy `"hola"`).


In [ ]:
import sys
import os
import json
import hashlib
import shutil
import importlib.util
import subprocess
from pathlib import Path

OUT = Path("/kaggle/working/out-eval29")
OUT.mkdir(parents=True, exist_ok=True)

STATE = {"env": {}, "stages": {}}

def log(*a):
    print("[eval29]", *a, flush=True)

def flush_state():
    (OUT / "manifest.json").write_text(json.dumps(STATE, ensure_ascii=False, indent=1), encoding="utf-8")

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while True:
            chunk = f.read(4 * 1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

# Deps defensivas
for mod_name, pip_name in [("einops", "einops"), ("yaml", "pyyaml"), ("triton", "triton"), ("huggingface_hub", "huggingface_hub")]:
    if importlib.util.find_spec(mod_name) is None:
        log(f"Instalando {pip_name}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

import torch
import triton

n_gpu = torch.cuda.device_count()
gpu_names = [torch.cuda.get_device_name(i) for i in range(n_gpu)]
STATE["env"] = {
    "python": sys.version,
    "torch": torch.__version__,
    "triton": getattr(triton, "__version__", "unknown"),
    "n_gpu": n_gpu,
    "gpu": gpu_names,
}
log("ENV:", STATE["env"])
assert n_gpu >= 1, f"Se requiere al menos 1 GPU, encontrados: {n_gpu}"

WORK = None
for cand_str in ("/kaggle/temp/eval29", "/root/eval29"):
    cand = Path(cand_str)
    cand.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(cand).free >= 20 * (1024 ** 3):
        WORK = cand
        break

assert WORK is not None, "No se encontro workdir con >= 20 GiB libres en /kaggle/temp/eval29 o /root/eval29"
log(f"WORK dir: {WORK} (free: {shutil.disk_usage(WORK).free / (1024**3):.2f} GiB)")
STATE["work_dir"] = str(WORK)

flush_state()


In [ ]:
# CELL 2 — Stage del dataset desde /kaggle/input (layout agnostico: zip / tar / carpeta)
import glob, zipfile, tarfile

# 1. Localizar raiz del dataset de forma robusta
cands = glob.glob("/kaggle/input/**/merge_lora.py", recursive=True)
assert cands, "no encontre merge_lora.py en /kaggle/input"
SRC = Path(cands[0]).parent
log(f"SRC detectado: {SRC}")

# 2. Arbol de /kaggle/input (hasta 3 niveles, ~60 entradas)
print("arbol de /kaggle/input:")
vistos = 0
for _root, _dirs, _files in os.walk("/kaggle/input", followlinks=True):
    _prof = _root.replace("/kaggle/input", "").count("/")
    if _prof > 3:
        _dirs[:] = []
        continue
    for _n in sorted(_files)[:8]:
        print("   ", os.path.join(_root, _n))
        vistos += 1
    for _d in sorted(_dirs)[:8]:
        print("   ", os.path.join(_root, _d) + "/")
    if vistos > 60:
        break

# 3. PROJ
PROJ = WORK / "kateto"
PROJ.mkdir(parents=True, exist_ok=True)

# 4. Extraer *.zip recursivo
zips = sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True))
for z in zips:
    base = os.path.basename(z)[:-4]
    with zipfile.ZipFile(z) as zf:
        names = zf.namelist()
    top = names[0].split("/")[0] if names else ""
    dest = str(PROJ) if top == base else os.path.join(str(PROJ), base)
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(dest)
    print(f"unzip {os.path.basename(z)} -> {dest} ({len(names)} entradas)")

# 5. Extraer *.tar recursivo
tars = sorted(glob.glob("/kaggle/input/**/*.tar", recursive=True))
for t in tars:
    with tarfile.open(t) as tf:
        names = tf.getnames()
    top = names[0].split("/")[0] if names else ""
    dest = str(PROJ) if top == os.path.basename(t)[:-4] else os.path.join(str(PROJ), top)
    os.makedirs(dest, exist_ok=True)
    with tarfile.open(t) as tf:
        tf.extractall(dest if top == os.path.basename(t)[:-4] else str(PROJ))
    print(f"untar {os.path.basename(t)} -> {dest} ({len(names)} entradas)")

# 6. Fallback: copiar archivos sueltos si faltan rutas requeridas o no habia archives
REQ_PATHS = [
    "scripts/eval_behavior.py",
    "scripts/eval_cbb_pmi.py",
    "scripts/eval_generation.py",
    "rwkv_pipeline/opd_rollout.py",
    "rwkv_pipeline/infer_kateto.py",
    "rwkv_pipeline/chat_template.py",
    "rwkv_pipeline/_stop_helpers.py",
    "RWKV-PEFT/rwkvt/rwkv7/model.py",
    "RWKV-PEFT/rwkv_vocab_v20230424.txt",
    "data/hf/che-boludo-benchmark/datasets/dataset_cbb_dpo.json",
    "out/eval/prompts_v1.json",
    "out/kateto-final/adapter_model.safetensors",
    "out/kateto-final/adapter_config.json",
    "config/behavior_patterns.yaml",
    "merge_lora.py",
    "data/eval_prompts.jsonl",
]
if not zips and not tars or any(not (PROJ / r).exists() for r in REQ_PATHS):
    log("Copiando archivos sueltos desde SRC...")
    for _p in glob.glob(str(SRC) + "/**/*", recursive=True):
        if os.path.isfile(_p):
            _rel = os.path.relpath(_p, str(SRC))
            _dst = PROJ / _rel
            _dst.parent.mkdir(parents=True, exist_ok=True)
            if not _dst.exists():
                shutil.copy2(_p, _dst)

# 7. Asserts
for r in REQ_PATHS:
    assert (PROJ / r).exists(), f"falta {r}"
log(f"Staging OK: {len(REQ_PATHS)} rutas verificadas en {PROJ}")

# 8. Estado
STATE["staging"] = {"src": str(SRC), "proj": str(PROJ), "n_checked": len(REQ_PATHS), "zips": len(zips), "tars": len(tars), "dataset_slug": "kateto-eval29"}
flush_state()


In [ ]:
from huggingface_hub import hf_hub_download

base_repo = "BlinkDL/rwkv7-g1"
base_filename = "rwkv7-g1j-2.9b-20260831-ctx16384.pth"
hf_dir = WORK / "hf"
hf_dir.mkdir(parents=True, exist_ok=True)
base_cand = hf_dir / base_filename

if not (base_cand.exists() and base_cand.stat().st_size > 5_500_000_000):
    log(f"Descargando {base_filename} desde HuggingFace ({base_repo})...")
    dl_path = hf_hub_download(
        repo_id=base_repo,
        filename=base_filename,
        local_dir=str(hf_dir),
    )
    BASE = Path(dl_path)
else:
    log(f"Base ya presente: {base_cand} ({base_cand.stat().st_size} bytes)")
    BASE = base_cand

base_bytes = BASE.stat().st_size
log(f"Calculando sha256 de base {BASE} ({base_bytes} bytes)...")
base_sha256 = sha256_file(BASE)
log(f"Base sha256: {base_sha256}")

STATE["base"] = {
    "path": str(BASE),
    "bytes": base_bytes,
    "sha256": base_sha256,
    "hf": f"{base_repo}::{base_filename}",
}
flush_state()


In [ ]:
MERGED = WORK / "kateto-rwkv29-merged.pth"
ADAPTER_DIR = PROJ / "out/kateto-final"
ADAPTER_WEIGHTS = ADAPTER_DIR / "adapter_model.safetensors"

if not (MERGED.exists() and MERGED.stat().st_size > 9_000_000_000):
    log("Iniciando merge_lora.py sobre base en CPU...")
    merge_cmd = [
        sys.executable,
        str(PROJ / "merge_lora.py"),
        str(BASE),
        str(ADAPTER_DIR),
        str(MERGED),
    ]
    res = subprocess.run(merge_cmd, cwd=str(PROJ))
    assert res.returncode == 0, f"merge_lora.py fallo con returncode={res.returncode}"
else:
    log(f"Merged ya existe con tamano suficiente: {MERGED} ({MERGED.stat().st_size} bytes)")

merged_bytes = MERGED.stat().st_size
log("Calculando sha256 de adapter...")
adapter_sha256 = sha256_file(ADAPTER_WEIGHTS)
log("Calculando sha256 de merged...")
merged_sha256 = sha256_file(MERGED)

STATE["adapter_sha256"] = adapter_sha256
STATE["merged"] = {
    "path": str(MERGED),
    "bytes": merged_bytes,
    "sha256": merged_sha256,
    "recipe": f"{BASE.name} + {ADAPTER_DIR.name} via merge_lora.py",
}

print(f"SHA256 base:    {STATE['base']['sha256']}")
print(f"SHA256 adapter: {STATE['adapter_sha256']}")
print(f"SHA256 merged:  {STATE['merged']['sha256']}")

flush_state()


In [ ]:
import time

def run_stage(name, argv, expect=()):
    log(f"STAGE {name} comenzando...")
    t0 = time.time()
    try:
        proc = subprocess.run(argv, cwd=str(PROJ))
        rc = proc.returncode
    except Exception as e:
        log(f"Excepcion corriendo stage {name}: {e}")
        rc = -1
    wall_s = round(time.time() - t0, 3)

    expect_paths = [Path(p) for p in expect]
    expect_ok = all(p.exists() for p in expect_paths)
    ok = (rc == 0) and expect_ok

    print(f"STAGE {name} rc={rc} wall={wall_s} ok={ok}", flush=True)
    if not expect_ok:
        missing = [str(p) for p in expect_paths if not p.exists()]
        log(f"Archivos esperados faltantes en {name}: {missing}")

    STATE["stages"][name] = {
        "argv": [str(x) for x in argv],
        "rc": rc,
        "wall_s": wall_s,
        "ok": ok,
    }
    flush_state()
    return ok

flush_state()


In [ ]:
pmi_chatml_out = OUT / "pmi29_chatml.json"
pmi_chatml_rows = OUT / "pmi29_chatml_rows.jsonl"

run_stage(
    "pmi_chatml",
    [
        sys.executable,
        str(PROJ / "scripts/eval_cbb_pmi.py"),
        "--ckpt", str(MERGED),
        "--chat-format", "chatml",
        "--n-pairs", "808",
        "--device", "cuda",
        "--out", str(pmi_chatml_out),
        "--dump-rows", str(pmi_chatml_rows),
    ],
    expect=[pmi_chatml_out],
)

if pmi_chatml_out.exists():
    d = json.loads(pmi_chatml_out.read_text(encoding="utf-8"))
    STATE["pmi_chatml"] = {
        "accuracy_raw": d.get("accuracy_raw"),
        "accuracy_pmi": d.get("accuracy_pmi"),
        "n_scored": d.get("n_scored"),
        "mean_logp": d.get("mean_logp"),
    }
    log("STATE[pmi_chatml]:", STATE["pmi_chatml"])
else:
    log("WARN: pmi29_chatml.json no encontrado")

flush_state()


In [ ]:
pmi_world_out = OUT / "pmi29_world.json"
pmi_world_rows = OUT / "pmi29_world_rows.jsonl"

run_stage(
    "pmi_world",
    [
        sys.executable,
        str(PROJ / "scripts/eval_cbb_pmi.py"),
        "--ckpt", str(MERGED),
        "--chat-format", "world",
        "--n-pairs", "808",
        "--device", "cuda",
        "--out", str(pmi_world_out),
        "--dump-rows", str(pmi_world_rows),
    ],
    expect=[pmi_world_out],
)

if pmi_world_out.exists():
    d = json.loads(pmi_world_out.read_text(encoding="utf-8"))
    STATE["pmi_world"] = {
        "accuracy_raw": d.get("accuracy_raw"),
        "accuracy_pmi": d.get("accuracy_pmi"),
        "n_scored": d.get("n_scored"),
        "mean_logp": d.get("mean_logp"),
    }
    log("STATE[pmi_world]:", STATE["pmi_world"])
else:
    log("WARN: pmi29_world.json no encontrado")

flush_state()


In [ ]:
behavior_out = OUT / "behavior29"
behavior_out.mkdir(parents=True, exist_ok=True)

run_stage(
    "behavior29",
    [
        sys.executable,
        str(PROJ / "scripts/eval_behavior.py"),
        "--baseline", str(BASE),
        "--candidate", str(MERGED),
        "--stages", "probes,cbb",
        "--device", "cuda",
        "--out-dir", str(behavior_out),
    ],
    expect=[behavior_out / "probes.json"],
)

flush_state()


In [ ]:
import statistics

pv1_path = PROJ / "out/eval/prompts_v1.json"
pv1_data = json.loads(pv1_path.read_text(encoding="utf-8"))
items = []
for it in pv1_data["generation"][:30]:
    items.append({
        "set": "prompts_v1_generation",
        "id": it["id"],
        "prompt": it["prompt"],
    })

ep_path = PROJ / "data/eval_prompts.jsonl"
ep_lines = [ln.strip() for ln in ep_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
for i, ln in enumerate(ep_lines[:30]):
    row_data = json.loads(ln)
    items.append({
        "set": "eval_prompts_jsonl",
        "id": f"ep_{i:02d}",
        "prompt": row_data["prompt"],
    })

if str(PROJ) not in sys.path:
    sys.path.insert(0, str(PROJ))
import rwkv_pipeline.opd_rollout as orl
from scripts.eval_behavior import load_student, ensure_eval
from chat_template import render_chat

model, tok, info = load_student(str(MERGED), device="cuda")
ensure_eval(model)
STATE["arch"] = info

def fmt(p, cf):
    return f"User: {p}\n\nAssistant:" if cf == "world" else render_chat([], p, "seco")

_ORIG = orl._next_logits

def make_timer():
    box = {"n": 0, "t0": None, "ttfab": None}
    def wrapped(*args, **kwargs):
        if box["n"] == 0:
            box["t0"] = time.time()
        res = _ORIG(*args, **kwargs)
        if box["n"] == 0:
            box["ttfab"] = round(time.time() - box["t0"], 4)
        box["n"] += 1
        return res
    return box, wrapped

def check_degenerate(text):
    t = text.strip()
    if len(t) < 8:
        return False, 0
    reps = 0
    for n in (8, 12, 16, 24):
        for i in range(0, max(0, len(t) - 2 * n)):
            sub = t[i:i + n]
            c = t.count(sub)
            if c > reps:
                reps = c
    return (reps >= 3), reps

gen_rows_file = OUT / "gen29_rows.jsonl"
dev_str = "cuda0:" + torch.cuda.get_device_name(0)

def gen_row(item, turno):
    box, wrapped = make_timer()
    orl._next_logits = wrapped
    try:
        torch.manual_seed(1337)
        prompt_txt = item["prompt"]
        prompt_formatted = fmt(prompt_txt, "chatml")
        r = orl.rollout_from_prompt(
            model, tok, prompt_formatted,
            max_len=64, temperature=0.80, top_p=0.70
        )
        text = orl._decode_str(tok, list(r.response_ids))
        is_deg, max_rep = check_degenerate(text)
        row = {
            "set": item["set"],
            "id": item["id"],
            "prompt": prompt_txt,
            "prompt_sha256": hashlib.sha256(prompt_txt.encode("utf-8")).hexdigest(),
            "model": "kateto-rwkv29-merged.pth",
            "motor": "rwkv-peft nativo (.pth) / rwkvt RWKV7, WKV=triton",
            "engine": "rwkvt.rwkv7.model.RWKV7",
            "device": dev_str,
            "turno": turno,
            "chat_format": "chatml",
            "ttfab_s": box["ttfab"],
            "wall_s": r.wall_s,
            "n_tokens": len(r.response_ids),
            "finished": bool(r.finished),
            "degenerate": is_deg,
            "max_repeat": max_rep,
            "text": text[:400],
            "error": None,
        }
    except Exception as e:
        prompt_txt = item["prompt"]
        row = {
            "set": item["set"],
            "id": item["id"],
            "prompt": prompt_txt,
            "prompt_sha256": hashlib.sha256(prompt_txt.encode("utf-8")).hexdigest(),
            "model": "kateto-rwkv29-merged.pth",
            "motor": "rwkv-peft nativo (.pth) / rwkvt RWKV7, WKV=triton",
            "engine": "rwkvt.rwkv7.model.RWKV7",
            "device": dev_str,
            "turno": turno,
            "chat_format": "chatml",
            "ttfab_s": box["ttfab"],
            "wall_s": None,
            "n_tokens": 0,
            "finished": False,
            "degenerate": False,
            "max_repeat": 0,
            "text": "",
            "error": str(e),
        }
    finally:
        orl._next_logits = _ORIG

    with gen_rows_file.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
    return row

if gen_rows_file.exists():
    gen_rows_file.unlink()

all_rows = []

dummy_item = {"set": "dummy", "id": "dummy_00", "prompt": "hola"}
r_dummy = gen_row(dummy_item, "frio_proceso")
all_rows.append(r_dummy)

for idx, it in enumerate(items):
    rf = gen_row(it, "frio")
    rc = gen_row(it, "caliente")
    all_rows.extend([rf, rc])
    if (idx + 1) % 10 == 0:
        log(f"Generados {idx + 1}/{len(items)} items...")

val_frio = [r["ttfab_s"] for r in all_rows if r["turno"] == "frio" and r["ttfab_s"] is not None and not r["error"]]
val_caliente = [r["ttfab_s"] for r in all_rows if r["turno"] == "caliente" and r["ttfab_s"] is not None and not r["error"]]

med_frio = round(statistics.median(val_frio), 4) if val_frio else None
med_caliente = round(statistics.median(val_caliente), 4) if val_caliente else None

STATE["gen_rows"] = {
    "n": len(all_rows),
    "median_ttfab_frio": med_frio,
    "median_ttfab_caliente": med_caliente,
}
log("STATE[gen_rows]:", STATE["gen_rows"])

from scripts.eval_behavior import free_gpu
free_gpu(model)

flush_state()


In [ ]:
gen_chatml_out = OUT / "gen29_metrics_chatml.json"

run_stage(
    "gen_metrics_chatml",
    [
        sys.executable,
        str(PROJ / "scripts/eval_generation.py"),
        "--backend", "local",
        "--ckpt", str(MERGED),
        "--chat-format", "chatml",
        "--prompts", str(PROJ / "out/eval/prompts_v1.json"),
        "--patterns", str(PROJ / "config/behavior_patterns.yaml"),
        "--n-items", "30",
        "--max-len", "64",
        "--temperature", "0.80",
        "--top-p", "0.70",
        "--seed", "1337",
        "--repeat-penalty", "1.30",
        "--out", str(gen_chatml_out),
    ],
    expect=[gen_chatml_out],
)

flush_state()


In [ ]:
gen_world_out = OUT / "gen29_metrics_world.json"

run_stage(
    "gen_metrics_world",
    [
        sys.executable,
        str(PROJ / "scripts/eval_generation.py"),
        "--backend", "local",
        "--ckpt", str(MERGED),
        "--chat-format", "world",
        "--prompts", str(PROJ / "out/eval/prompts_v1.json"),
        "--patterns", str(PROJ / "config/behavior_patterns.yaml"),
        "--n-items", "30",
        "--max-len", "64",
        "--temperature", "0.80",
        "--top-p", "0.70",
        "--seed", "1337",
        "--repeat-penalty", "1.30",
        "--out", str(gen_world_out),
    ],
    expect=[gen_world_out],
)

flush_state()


In [ ]:
def load_json_safe(p):
    try:
        p = Path(p)
        if p.exists():
            return json.loads(p.read_text(encoding="utf-8"))
    except Exception as e:
        log(f"Error leyendo {p}: {e}")
    return {}

manifest = load_json_safe(OUT / "manifest.json")
pmi_chatml = load_json_safe(OUT / "pmi29_chatml.json")
pmi_world = load_json_safe(OUT / "pmi29_world.json")
probes_data = load_json_safe(OUT / "behavior29/probes.json")
gen_chatml = load_json_safe(OUT / "gen29_metrics_chatml.json")
gen_world = load_json_safe(OUT / "gen29_metrics_world.json")

env_info = manifest.get("env", {})
n_gpu = env_info.get("n_gpu", "unknown")
torch_ver = env_info.get("torch", "unknown")
triton_ver = env_info.get("triton", "unknown")

sha_base = manifest.get("base", {}).get("sha256", "unknown")
sha_adapter = manifest.get("adapter_sha256", "unknown")
sha_merged = manifest.get("merged", {}).get("sha256", "unknown")

device_info = env_info.get("gpu", ["unknown"])

pmi_chatml_raw = pmi_chatml.get("accuracy_raw", "N/A")
pmi_chatml_pmi = pmi_chatml.get("accuracy_pmi", "N/A")
pmi_world_raw = pmi_world.get("accuracy_raw", "N/A")
pmi_world_pmi = pmi_world.get("accuracy_pmi", "N/A")

formatgen_choice = probes_data.get("formatgen", {}).get("chosen_format", "N/A")

vice_chatml = gen_chatml.get("assistant_vice", {})
vice_world = gen_world.get("assistant_vice", {})
degen_chatml = gen_chatml.get("degeneracy", {})
degen_world = gen_world.get("degeneracy", {})

gen_rows_info = manifest.get("gen_rows", {})
med_ttfab_frio = gen_rows_info.get("median_ttfab_frio", "N/A")
med_ttfab_caliente = gen_rows_info.get("median_ttfab_caliente", "N/A")

out_files_lines = []
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        size_bytes = p.stat().st_size
        out_files_lines.append(f"- `{p.relative_to(OUT)}`: {size_bytes} bytes ({size_bytes / (1024*1024):.2f} MB)")

summary_content = f"""# Resumen de Evaluacion RWKV 2.9B Merged (eval29)

## Entorno
- **GPUs**: {n_gpu} ({device_info})
- **PyTorch**: {torch_ver}
- **Triton**: {triton_ver}

## Hashes SHA-256
- **Base**: `{sha_base}`
- **Adapter**: `{sha_adapter}`
- **Merged**: `{sha_merged}`

## CBB PMI Accuracy
- **ChatML**: raw = `{pmi_chatml_raw}`, pmi = `{pmi_chatml_pmi}`
- **World**:  raw = `{pmi_world_raw}`, pmi = `{pmi_world_pmi}`

## Sondas de Comportamiento
- **Eleccion de formato probe generacion (formatgen)**: `{formatgen_choice}`

## Metricas de Generacion (Repo)
### ChatML
- **assistant_vice**: {json.dumps(vice_chatml, ensure_ascii=False)}
- **degeneracy**: {json.dumps(degen_chatml, ensure_ascii=False)}

### World
- **assistant_vice**: {json.dumps(vice_world, ensure_ascii=False)}
- **degeneracy**: {json.dumps(degen_world, ensure_ascii=False)}

## TTFAB (In-Process)
- **Mediana TTFAB frio**: `{med_ttfab_frio}` s
- **Mediana TTFAB caliente**: `{med_ttfab_caliente}` s

## Archivos de Salida
{chr(10).join(out_files_lines)}
"""

(OUT / "SUMMARY.md").write_text(summary_content, encoding="utf-8")
log(f"SUMMARY.md escrito en {OUT / 'SUMMARY.md'}")

print("eval29 DONE", flush=True)
flush_state()
